In [ ]:
%reset -f

In [ ]:
# Hyper parameters
batch_size = 64
learning_rate = 1e-3
epochs = 40

In [ ]:
import torch
# import torch.accelerator
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats
import os
import pandas as pd
from typing import Callable

In [ ]:
try:
    device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
except AttributeError:
    device = "cpu"
device

In [ ]:
def list_from_str(str_list: str, fn_apply_to_items: Callable) -> list:
    if str_list in ["", "[]"]:
        return []

    elems = str_list.strip("[]\"").split(", ")
    return list(map(fn_apply_to_items, elems))

In [ ]:
def normalize_mean_list(means_list: list[float]) -> dict[str, int | float]:
    means = torch.tensor(means_list, dtype=torch.float)

    assert means.dim() == 1

    if means.numel() == 0:
        return {
            "mean": 0,
            "median": 0,
            "range": 0,
            "std": 0,
            "var": 0,
            "medianAd": 0,
            "meanAd": 0,
        }

    mean = means.mean(0)
    median = means.median()
    range = means.max().item() - means.min().item()

    std = 0 if means.numel() == 1 else means.std(0).item()
    var = std ** 2
    median_ad = float(scipy.stats.median_abs_deviation(means.numpy()))
    mean_ad = means.sub(mean).absolute().mean()

    assert isinstance(median, torch.Tensor)

    return {
        "mean": mean.item(),
        "median": median.item(),
        "range": range,
        "std": std,
        "var": var,
        "medianAd": median_ad,
        "meanAd": mean_ad.item(),
    }

normalize_mean_list([2, 2, 3, 4, 14])

In [ ]:
from torch._tensor import Tensor


class PIIStateDataset(Dataset):
    def __init__(self, data_file: str) -> None:
        df = pd.read_csv(data_file)

        self.numUnstable = torch.tensor(df["numUnstable"], dtype=torch.float)
        self.numNM1 = torch.tensor(df["numNM1"], dtype=torch.float)
        self.numNM2 = torch.tensor(df["numNM2"], dtype=torch.float)

        start, end = df.columns.slice_locs("matchingAM","nm2RAM")
        mean_columns = df.iloc[:, start:end]
        self.meanData = {}

        for col in mean_columns:
            meanDicts = [normalize_mean_list(list_from_str(means, float)) for means in df[col]]

            for key in meanDicts[0].keys():
                valueList = [meanDict[key] for meanDict in meanDicts]
                self.meanData[col + key.title()] = torch.tensor(valueList, dtype=torch.float)

        self.numEdges = torch.tensor(df["numEdges"], dtype=torch.float)
        self.numSingletons = torch.tensor(df["numSingletons"], dtype=torch.float)
        self.numChains = torch.tensor(df["numChains"], dtype=torch.float)
        self.numCycles = torch.tensor(df["numCycles"], dtype=torch.float)

        self.avgChainLen = torch.tensor(df["avgChainLen"], dtype=torch.float)
        self.avgCycleLen = torch.tensor(df["avgCycleLen"], dtype=torch.float)

        self.converges = torch.tensor(df["converges"], dtype=torch.long)
        self.convergesOneHot = nn.functional.one_hot(self.converges, 2).float()

        self.singleton_features = torch.stack((
            self.numUnstable, self.numNM1, self.numNM2,
            self.numEdges, self.numSingletons, self.numChains, self.numCycles,
            self.avgChainLen, self.avgCycleLen
        ), dim=1)
        self.means_features = torch.stack(tuple(self.meanData.values()), dim=1)

        self.features = torch.cat((self.singleton_features, self.means_features), dim=1)

        display(self.features)
        display(self.features.shape)

    def __len__(self):
        return len(self.converges)

    def __getitem__(self, idx) -> tuple[Tensor, Tensor]:
        return self.features[idx], self.convergesOneHot[idx]

In [ ]:
# dataset = "stateData_2000_10"
dataset = "stateData_2000_10_itr0"
# dataset = "stateData_2000_10_itr1"
# dataset = "stateData_2000_10_itr2"
# dataset = "stateData_2000_10_itr_-1"

# dataset = "balStateData_20000_10_itr0"
# dataset = "balStateData_20000_10_itr1"
# dataset = "balStateData_20000_10_itr2"
# dataset = "balStateData_20000_10_itr_-1"

training_data = PIIStateDataset(f"data/{dataset}.csv")
test_data = PIIStateDataset(f"data/{dataset}_test.csv")

train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [ ]:
training_data[0][0].shape

In [ ]:
for i in range(10):
    print(i)
    print(training_data[i])

In [ ]:
# # Check overlap between training and testing
# df_train = pd.read_csv(f"sign_{n}_{perm_total}.csv")
# df_test = pd.read_csv(f"sign_{n}_{perm_total}_test.csv")

# set(df_train["permutation"]).intersection(set(df_test["permutation"]))

In [ ]:
%reset_selective -f (model|loss_fn|optimizer)

In [ ]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()

        self.linear_relu_stack = nn.Sequential(
            nn.Linear(training_data[0][0].size(0), 64),
            nn.ReLU(),
            nn.Linear(64, 16),
            nn.ReLU(),
            # nn.Linear(16, 16),
            # nn.ReLU(),
            nn.Linear(16, 2),
            # nn.Softmax()
            # nn.ReLU()
        )

    def forward(self, x):
        # x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
model = NeuralNetwork().to(device)

In [ ]:
# loss_fn = nn.CrossEntropyLoss()
# loss_fn = nn.BCELoss()
loss_fn = nn.BCEWithLogitsLoss()

# optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# optimizer = torch.optim.ASGD(model.parameters(), lr=learning_rate)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
# optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    train_loss, correct = 0, 0

    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # X = X.to(device)
        # y = y.to(device)

        # Compute prediction and loss
        pred = model(X)

        # if batch == 0:
        #     print(pred)
        #     print(y, y.shape)
        #     print(pred.argmax(1).eq(y.argmax(1)))
        #     print(pred.argmax(1).eq(y.argmax(1)).sum().item())

        loss = loss_fn(pred, y)
        train_loss += loss.item()

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # Print out losses every 100 batches
        # if batch % 100 == 0:
        #     loss, current = loss.item(), batch * batch_size + len(X)
        #     print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

        correct += pred.argmax(1).eq(y.argmax(1)).sum().item()

    correct /= size
    train_loss /= num_batches

    # print(f"Train Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {train_loss:>8f} \n")

    return train_loss, correct


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # NOTE: This is for sanity checking the argmax tensor accuracy arithmetic
    # sanity_correct = 0
    # sanity_num_tests = 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for batch, (X, y) in enumerate(dataloader):
            # X = X.to(device)
            # y = y.to(device)

            assert isinstance(y, torch.Tensor)

            pred: torch.Tensor = model(X)
            test_loss += loss_fn(pred, y).item()

            # if batch < 5:
            #     print(pred)
            #     print(y)
            #     print("---")
            #     print(pred.argmax(1))
            #     print(y.argmax(1))
            #     print("---")
            #     print(pred.argmax(1).eq(y.argmax(1)).sum().item())

            # NOTE: Part of the sanity checking accuracy
            # for row, label in zip(pred, y):
            #     # print(row, label)
            #     # print(row.argmax(0), label.argmax(0))
            #     if row.argmax(0).item() == label.argmax(0).item():
            #         sanity_correct += 1
            #     sanity_num_tests += 1

            # pred = pred.round()
            correct += pred.argmax(1).eq(y.argmax(1)).sum().item()


    test_loss /= num_batches
    correct /= size
    # print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    # print(f"SANITY Accuracy: {100 * sanity_correct / sanity_num_tests:>0.1f}%")

    return test_loss, correct

In [ ]:
train_losses, train_accuracies = np.zeros(epochs, dtype=np.float32), np.empty(epochs, dtype=np.float32)
test_losses, test_accuracies = np.zeros(epochs, dtype=np.float32), np.empty(epochs, dtype=np.float32)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loss, train_accuracy = train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loss, test_accuracy = test_loop(test_dataloader, model, loss_fn)

    train_losses[t] = train_loss
    train_accuracies[t] = train_accuracy
    test_losses[t] = test_loss
    test_accuracies[t] = test_accuracy

    print(f"Train Error: \n Accuracy: {(100*train_accuracy):>0.1f}%, Avg loss: {train_loss:>8f} \n")
    print(f"Test Error: \n Accuracy: {(100*test_accuracy):>0.1f}%, Avg loss: {test_loss:>8f} \n")
print("Done!")

In [ ]:
def plot_data(
        x, ys: np.ndarray, labels: list[str] | None = None, title: str = "", ylabel: str = ""
    ) -> tuple:
    fig, ax = plt.subplots()

    if len(ys.shape) > 1:
        assert isinstance(labels, list)
        for y in ys:
            ax.plot(x, y)
    else:
        line = ax.plot(x, ys)
        ax.legend(handles=line)

    ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
    ax.grid()

    if labels != None:
        ax.legend(labels)

    return fig, ax

In [ ]:
x = np.arange(0, epochs)

losses = np.vstack((train_losses, test_losses))
loss_fig, loss_ax = plot_data(x, losses, ["Training", "Testing"], f"Loss vs. Epoch ({dataset})", "Loss")
plt.savefig(f"plots/{dataset}_loss")

accuracies = np.vstack((train_accuracies, test_accuracies)) * 100
acc_fig, acc_ax = plot_data(x, accuracies, ["Training", "Testing"], f"Accuracy vs. Epoch ({dataset})", "Accuracy (%)")
plt.savefig(f"plots/{dataset}_acc")